In [22]:
######################
# Calculate CAPE
# 2026.4.27
# Mu-Ting Chien
######################
import numpy as np
import xarray as xr
import os
from metpy.calc import cape_cin, dewpoint_from_relative_humidity, parcel_profile
from metpy.units import units

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    # filename="aquaplanet_logs/cape_calculation.log",      # <— write logs here
    # filemode="a",               # "a" = append (default), "w" = overwrite
)

logger = logging.getLogger(__name__)


##############
# 1. Load data
#################
dir_in = '/glade/campaign/univ/uwas0114/processed_pcoord/'
expname_list = list(['-4K','CTL','4K'])
fout_list = list(['m4K','ctl','p4K'])

Ptop_all = np.array([75, 100, 125])*100 # Find tropoppause for each exp!

# for iexp in range(0, 3):#3): # 0, 3
iexp = 2
    
dir_in_sub = dir_in + fout_list[iexp]+'/'

# for iyr in range(3, 6):#6): # 3, 6
iyr = 3

yy_str = f"{iyr:04d}"
logger.info('Start calculating CAPE for :'+ expname_list[iexp]+','+yy_str)

# Load T
fname = dir_in_sub + fout_list[iexp]+'_T_yr'+yy_str+'.nc'
ds    = xr.open_dataset(fname).sel(lat=slice(-15,15))
T = ds['T']
time = ds['time']
lat  = ds['lat']
lon  = ds['lon']
plev = ds['plev']
logger.info('T loaded')

# Load q
fname = dir_in_sub + fout_list[iexp]+'_Q_yr'+yy_str+'.nc'
ds    = xr.open_dataset(fname).sel(lat=slice(-15,15))
q = ds['Q']
logger.info('Q loaded')

# Define CC equation
epsilon = 0.622
e = 6.1094*np.exp( 17.625*(T-273.15)/(T-273.15+243.04) ) # hPa
plev_large = np.tile(plev, (np.size(T,0), np.size(T, 1), np.size(T, 2), 1))
print(np.shape(plev_large))
#for ilev in range(0, np.size(plev)):
#    if ilev == 0:
#        RH = np.empty([np.size(T,0), np.size(T, 1), np.size(T, 2), np.size(T, 3)])
RH = q/( epsilon*e/plev_large )
logger.info("RH calculated")

del q
# Calculate CAPE
RH_r = RH[:,:,:,::-1] * units.dimensionless
del RH
T_C = (T[:,:,:,::-1] - 273.15) * units.degC # unit C
plev_r = plev[::-1] * units.hPa
plev_large_r = plev_large[:,:,:,::-1] * units.hPa

del T

# calculate dewpoint
Td = dewpoint_from_relative_humidity(T_C, RH_r)
# del RH_r
logger.info("Dewpoint calculated")

2026-04-28 14:49:55,769 [INFO] Start calculating CAPE for :4K,0003
2026-04-28 14:49:55,873 [INFO] T loaded
2026-04-28 14:49:55,937 [INFO] Q loaded


(2920, 16, 144, 40)


2026-04-28 14:50:01,573 [INFO] RH calculated
/glade/derecho/scratch/sressel/tmp/ipykernel_114151/1335361955.py:82: UserWarning: Relative humidity >120%, ensure proper units.
  Td = dewpoint_from_relative_humidity(T_C, RH_r)
/glade/u/home/sressel/.conda/envs/modified-npl/lib/python3.12/site-packages/metpy/calc/thermo.py:1403: RuntimeWarning: divide by zero encountered in log
  val = np.log(vapor_pressure / mpconsts.nounit.sat_pressure_0c)
/glade/u/home/sressel/.conda/envs/modified-npl/lib/python3.12/site-packages/metpy/calc/thermo.py:1404: RuntimeWarning: invalid value encountered in divide
  return mpconsts.nounit.zero_degc + 243.5 * val / (17.67 - val)
2026-04-28 14:50:09,056 [INFO] Dewpoint calculated


In [71]:
fname = dir_in_sub + fout_list[iexp]+'_T_yr'+yy_str+'.nc'
ds    = xr.open_dataset(fname).sel(lat=slice(-15,15))
T = ds['T']

fname = dir_in_sub + fout_list[iexp]+'_Q_yr'+yy_str+'.nc'
ds    = xr.open_dataset(fname).sel(lat=slice(-15,15))
q = ds['Q']
logger.info('Q loaded')

# Define CC equation
epsilon = 0.622
e = 6.1094*np.exp( 17.625*(T-273.15)/(T-273.15+243.04) ) # hPa
plev_large = np.tile(plev, (np.size(T,0), np.size(T, 1), np.size(T, 2), 1))
print(np.shape(plev_large))
#for ilev in range(0, np.size(plev)):
#    if ilev == 0:
#        RH = np.empty([np.size(T,0), np.size(T, 1), np.size(T, 2), np.size(T, 3)])
RH = q/( epsilon*e/plev_large )

test_RH = q/( epsilon*e/plev)

2026-04-28 15:07:30,856 [INFO] Q loaded


(2920, 16, 144, 40)


In [ ]:
from metpy.calc import saturation_vapor_pressure

test_e = saturation_vapor_pressure(T * units.K).metpy.convert_units('hPa')


In [98]:
print(np.nanmax(test_e.values -  e.values))

2.870140989539509e+71


In [4]:
T_parc = parcel_profile(plev_r, T_C[0,0,0,0], Td[0,0,0,0])#.to('degC')

In [ ]:
CAPE, CIN = xr.zeros_like(Td).isel(plev=-1, drop=True), xr.zeros_like(Td).isel(plev=-1, drop=True)

# compute parcel temperature
logger.info("Computing parcel temperatures...")
# for it in range(0, np.size(Td,0)):
for it in range(0, 3):
    if (it+1) % 1 == 0:
        logger.info(f"Hour index: ({3*(it+1)}/{3*np.size(Td,0)})")

    for ilat in range(0, np.size(Td, 1)):
        for ilon in range(0, np.size(Td, 2)):
            T_parc = parcel_profile(plev_r, T_C[it,ilat,ilon,0], Td[it,ilat,ilon,0])#.to('degC')
            T_parc_C = (T_parc.values - 273.15)* units.degC
            CAPE_col, CIN_col = cape_cin(plev_r, T_C[it,ilat,ilon], Td[it,ilat,ilon], T_parc_C)
            CAPE[it,ilat,ilon] = CAPE_col.magnitude
            CIN[it,ilat,ilon] = CIN_col.magnitude
#T_parc = parcel_profile(plev_r, T_C[:,:,:,0], Td[:,:,:,0]).to('degC')
logger.info("Parcel temperatures calculated")

2026-04-28 15:00:25,365 [INFO] Computing parcel temperatures...
2026-04-28 15:00:25,366 [INFO] Time index: (1/2920)
2026-04-28 15:00:46,008 [INFO] Time index: (2/2920)
2026-04-28 15:01:06,662 [INFO] Time index: (3/2920)
2026-04-28 15:01:27,476 [INFO] Parcel temperatures calculated


In [105]:
time[-1].values

array(2919)

In [110]:
for it in range(0, 3):
    if (it+1) % 1 == 0:
        logger.info(f"Day {3*(it+1)/24}/{3*np.size(Td,0)/24}")

2026-04-28 15:24:44,256 [INFO] Day 0.125/365.0
2026-04-28 15:24:44,257 [INFO] Day 0.25/365.0
2026-04-28 15:24:44,257 [INFO] Day 0.375/365.0


In [6]:
# calculate surface-based CAPE/CIN
T_parc_C = (T_parc.values - 273.15)* units.degC
CAPE, CIN = cape_cin(plev_r, T_C, Td, T_parc_C)
logger.info("CAPE, CIN calculated")

ValueError: non-broadcastable output operand with shape (40,) doesn't match the broadcast shape (2920,16,144,40)

In [20]:
type(Td)

xarray.core.dataarray.DataArray

In [117]:
da_CAPE = xr.DataArray(CAPE,\
    coords={"time": time, "lat": lat, "lon": lon},\
    dims=("time", "lat", "lon"), name="CAPE")

da_CIN = xr.DataArray(CIN,\
    coords={"time": time, "lat": lat, "lon": lon},\
    dims=("time", "lat", "lon"), name="CIN")

ds_out = xr.Dataset({"CAPE": (['time', 'lat', 'lon'], da_CAPE.data), "CIN": (['time', 'lat', 'lon'], da_CIN.data)},coords={'time': ds.time.values, 'lat': ds.lat.values, 'lon': ds.lon.values})

In [118]:
ds_out

<xarray.Dataset> Size: 108MB
Dimensions:  (time: 2920, lat: 16, lon: 144)
Coordinates:
  * time     (time) int64 23kB 0 1 2 3 4 5 6 ... 2914 2915 2916 2917 2918 2919
  * lat      (lat) float64 128B -14.21 -12.32 -10.42 ... 10.42 12.32 14.21
  * lon      (lon) float64 1kB 0.0 2.5 5.0 7.5 10.0 ... 350.0 352.5 355.0 357.5
Data variables:
    CAPE     (time, lat, lon) float64 54MB 1.153e+03 952.3 715.7 ... 0.0 0.0 0.0
    CIN      (time, lat, lon) float64 54MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0